## Solving CartPole with DeepQNetworks


In [ ]:
%load_ext tensorboard

## Deep Q-Networks

Deep Q-Networks take a state vector and learn long-term value estimates for each action in that state, known as Q-values. The Q-value $Q(s, a)$ represents the expected cumulative discounted reward from taking action $a$ in state $s$ and following the optimal policy thereafter.

Naively applying Q-learning to neural networks fails due to two problems:
1. **Bootstrap instability** — the TD target depends on the network's own predictions, creating a moving target.
2. **Correlated samples** — consecutive transitions from the same episode are highly correlated, violating the i.i.d. assumption that gradient descent relies on.

DQN (Mnih et al., 2015) introduced two mechanisms to address these issues: replay buffers and target networks.

![Alt text](best_run.png)

### Replay Buffers

Neural network gradients are computed over minibatches, which represent an average direction of movement. For stable learning, batches need a roughly even distribution of experiences — otherwise the network swings in one direction for one epoch and overshoots in the opposite direction the next.

If the agent computes the TD error at every step using only the current transition, it constantly overwrites what it previously learned. Each new state and episode pushes the weights in a new direction. The agent needs repeated exposure to past experiences to form stable gradients. This is the role of the replay buffer: at every step, the agent samples a random minibatch from its stored history, so it learns from a diverse set of transitions rather than a single correlated sequence.

### Target Networks

Even with replay buffers, the learning target is unstable. The TD target for Q-learning is:

$$y = r + \gamma \max_{a'} Q(s', a'; \theta)$$

Because the target depends on the same network parameters $\theta$ that are being updated, every gradient step shifts the target. This creates a feedback loop — the network is chasing a moving target, like a dog chasing its own tail.

DQN solves this with a separate **target network** $Q(s, a; \theta^{-})$ whose weights are frozen and only copied from the main network every $N$ steps:

$$y = r + \gamma \max_{a'} Q(s', a'; \theta^{-})$$

This works because the first values the network learns correctly are around terminal states, where the target is simply $r$ (no bootstrap). With each target network update, correct values propagate one step further from the terminal states. Neural networks are more sensitive to shifting distributions than to initially inaccurate targets — the inaccuracies get corrected over time as long as the learning signal is stable.

In [ ]:
import os
from typing import Any

import gymnasium as gym
import structlog
import torch
from cart_pole_logger import CartPoleLogger
from torch import nn

from samsara_rl.control.dqn.dqn_agent import (
    DQNAgent,
)
from samsara_rl.mdp.cart_pole.scaled_cart_pole import ScaledCartPole
from samsara_rl.policy.epsilon_greedy import DecaySchedule

log = structlog.get_logger()

In [ ]:
log = structlog.get_logger()

MAX_EPISODES = 2800
EVAL_EPISODES = 20
BASE_DIR = "logs/double_dqn_example"
ALPHAS = [0.0005, 0.0003, 0.001]  # 0.005, 0.001, 0.0005, 0.0001, 0.00005]
GAMMAS = [0.99]

In [ ]:
class CartPoleQNetwork(nn.Module):
    def __init__(self, input_dim: int, hidden_dim_1: int, hidden_dim_2: int, output_dim: int) -> None:
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim_1)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim_1, hidden_dim_2)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(hidden_dim_2, output_dim)

    def forward(self, x: Any) -> torch.Tensor:
        out = self.fc1(x)
        out = self.relu(out)
        out = self.fc2(out)
        out = self.relu2(out)
        return self.fc3(out)  # type: ignore[no-any-return]

In [ ]:
def run_experiment(env, alpha: float, gamma: float) -> None:
    """Train and evaluate a single configuration.

    Creates a LinearFunction and QLearningGradient agent, trains for
    MAX_EPISODES episodes, saves the learned weights, and runs greedy
    evaluation.

    Args:
        env: Gymnasium CartPole environment.
        alpha: Learning rate for the Q-learning update.
    """
    model_name = "deep_q_network"
    run_name = f"{model_name}_alpha={alpha}_gamma={gamma}"
    run_dir = os.path.join(BASE_DIR, run_name)
    os.makedirs(run_dir, exist_ok=True)

    log.info("starting_run", run=run_name)

    fc = CartPoleQNetwork(4, 32, 16, 2)  # FullyConnected(4, 32, 2,dtype=torch.float32)

    agent = DQNAgent(
        mdp=env,
        gamma=gamma,
        q=fc,
        alpha=alpha,
        log_dir=run_dir,
        experiment_name=run_name,
        lr_step_size=120,
        lr_gamma=0.3,
        epsilon_episodes=800,
        epsilon_schedule=DecaySchedule.LINEAR,
        batch_size=128,
        buffer_size=10000,
        target_update_freq=500,
    )

    agent.register(CartPoleLogger())

    agent.evaluate(max_iter=MAX_EPISODES)
    #     save_model(agent, run_dir)

    if agent.tensorboard:
        agent.tensorboard.flush()

In [ ]:
def main() -> None:
    """Run grid search over feature configs, bias, and learning rates.

    Iterates over all combinations of FEATURE_CONFIGS, BIAS_CONFIGS,
    and ALPHAS, training and evaluating each one.
    """
    env = ScaledCartPole(gym.make("CartPole-v1"))

    for alpha in ALPHAS:
        for gamma in GAMMAS:
            run_experiment(env, alpha, gamma)

In [ ]:
main()

## Analysis

### Early Training (Episode 20)

Initially the action advantage — $Q(\text{PushLeft}) - Q(\text{PushRight})$ — is largely random. The agent is exploring with a high $\epsilon$, which controls how often it takes a completely random action versus using the network's current best estimate.

![Alt text](action_advantage_ep20.png)

### Converged Policy (Episode 900)

The advantage surface shows the agent has learned to counter-steer:

- **Positive (blue):** $Q(\text{PushLeft}) > Q(\text{PushRight})$ — push left is preferred. This appears when the pole leans right (positive angle), which is correct.
- **Negative (red):** $Q(\text{PushRight}) > Q(\text{PushLeft})$ — push right is preferred. This appears when the pole leans left (negative angle).
- **Near zero (white):** Both actions are roughly equal — the pole is near upright and either action is fine.

![Alt text](action_advantage_ep900.png)

A well-trained agent shows a clean gradient from blue to red along the angle axis, meaning it has learned to consistently counter-steer. The magnitude indicates confidence — larger values mean the agent strongly prefers one action over the other in that region of state space.